# Notebook 2: A planning pattern you can watch think (ReAct)

ReAct means the model alternates a **thought** and an **action**, looking at the
result (an **observation**) before the next thought. It is the most important
pattern for tool-using agents.

In this notebook we build a small ReAct loop from scratch and print every
Thought, Action, and Observation, so you can literally watch the agent reason.

## Setup: run these first

Every notebook starts the same way: install the library, load your key, and
pick a model. Run the next three cells in order.

**Save yourself time:** store your API key once in Colab Secrets (the key icon on
the left sidebar). Add a secret named `ANTHROPIC_API_KEY`, paste your key as the
value, and turn on "Notebook access". After that, every notebook picks it up
automatically and you never paste it again.

In [ ]:
# 1) Install the Anthropic library (Colab forgets it between sessions, so we
#    install each time). The -q just keeps the output quiet.
!pip install -q anthropic

In [ ]:
# 2) Load your Anthropic API key.
#    Easiest: store it once in Colab Secrets (key icon on the left), named
#    ANTHROPIC_API_KEY, with "Notebook access" on. Then this reads it for you.
import anthropic

api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    api_key = None  # not on Colab, or no secret set yet

if not api_key:
    # Fallback: ask once if no secret is found. The input is hidden and not saved.
    from getpass import getpass
    api_key = getpass("Paste your Anthropic API key (input is hidden): ")

client = anthropic.Anthropic(api_key=api_key)
print("Your client is ready.")

In [ ]:
# 3) Choose the model. claude-opus-4-8 is the most capable. For faster, cheaper
#    runs while practising you can switch this to "claude-haiku-4-5".
MODEL = "claude-opus-4-8"
print("Using model:", MODEL)

## Step 1: the tools

We give the agent a fact lookup tool and the calculator from before.

In [ ]:
import ast
import operator

# A tool is described by a name, a plain description, and the inputs it needs.
# The model reads the description to decide when to use the tool.
calculator_tool = {
    "name": "calculator",
    "description": "Evaluate a basic arithmetic expression and return the result. Use this whenever the user asks for a calculation.",
    "input_schema": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "An arithmetic expression, for example '12 * 37'.",
            }
        },
        "required": ["expression"],
    },
}

# The safe operations our calculator allows: plus, minus, times, divide, and
# negative numbers. Nothing else is permitted.
_SAFE_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}


def _safe_eval(node):
    # Walk the parsed expression and only allow numbers and the safe operations
    # above. Anything else (names, function calls) raises an error, so no
    # arbitrary code can run. This is the safe alternative to eval().
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("unsupported expression")


def run_calculator(expression):
    # Beginners can treat this as a black box: give it an expression, get the
    # result. It parses the text and evaluates only safe arithmetic, so it
    # cannot run arbitrary code.
    try:
        tree = ast.parse(expression, mode="eval")
        return str(_safe_eval(tree))
    except Exception:
        return "Error: I could only handle a simple arithmetic expression like '12 * 37'."

In [ ]:
# Two small tools. One looks up an approximate fact, one does arithmetic.
facts = {
    "geneva population": "200000 (approximate, for the workshop)",
    "zurich population": "430000 (approximate, for the workshop)",
    "bern population": "135000 (approximate, for the workshop)",
}

lookup_tool = {
    "name": "lookup_fact",
    "description": "Look up an approximate fact by name, for example 'geneva population'. Use this when you need a number you do not know.",
    "input_schema": {
        "type": "object",
        "properties": {
            "name": {"type": "string", "description": "The fact to look up, lowercase."}
        },
        "required": ["name"],
    },
}


def run_lookup(name):
    return facts.get(name.lower().strip(), "Error: I do not have that fact. Try another name.")

In [ ]:
tools = [lookup_tool, calculator_tool]

# A small helper that runs whichever tool the model asked for.
def run_tool(name, tool_input):
    if name == "lookup_fact":
        return run_lookup(tool_input["name"])
    if name == "calculator":
        return run_calculator(tool_input["expression"])
    return "Error: unknown tool."

SYSTEM = (
    "You are a careful assistant. Think step by step. "
    "When you need a fact or a calculation, use the tools provided, one step at a time. "
    "After you have everything you need, give a short final answer."
)

## Step 2: the ReAct loop

The loop keeps going while the model asks for tools. Each pass it prints the
model's Thought, runs the Action, and prints the Observation. When the model
stops asking for tools, it has a final answer.

In [ ]:
# YOUR TURN. Complete the two marked lines. The instructor notebook has the answer.

def run_agent(question, max_steps=5):
    messages = [{"role": "user", "content": question}]
    for step in range(max_steps):
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            system=SYSTEM,
            tools=tools,
            messages=messages,
        )

        # Print the model's reasoning (its text) as a Thought.
        for block in response.content:
            if block.type == "text" and block.text.strip():
                print("Thought:", block.text.strip())

        # If the model is not asking for a tool, it is done.
        if response.stop_reason != "tool_use":
            for block in response.content:
                if block.type == "text":
                    return block.text
            return ""

        # Otherwise, run each requested tool and record the observations.
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print("Action:", block.name, block.input)
                # TODO: run the tool with run_tool(block.name, block.input) and save as `observation`.
                observation = None  # replace this line
                print("Observation:", observation)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": observation,
                })
        # TODO: add the tool_results back to messages as a user turn (see Notebook 1).
        pass  # replace this line

    return "Stopped after the maximum number of steps."

## Step 3: run it

This question needs two lookups and one calculation, so watch the loop work.

In [ ]:
answer = run_agent("What is the combined population of Geneva and Zurich? Use the tools.")
print("\nFinal answer:", answer)

## Recap

You watched the agent **think**, **act** (use a tool), and **observe**, over and
over, until it had everything it needed. That loop is ReAct.

## Your turn

Ask a question that needs the population of Bern as well, and see how the agent
plans across three lookups.